# AU-saliency Probe — can Colab run it, and how long?

Throwaway benchmark. Answers **(1)** does py-feat run on a Colab T4, and **(2)** how many seconds/clip AU-on `z_v` extraction takes -> real total for the full dataset.

**Non-destructive:** only calls `get_z_v()` (returns a tensor, writes nothing) on a few sample clips in `/content`. It never touches any feature cache.

### How to use
1. Runtime -> Change runtime type -> **T4 GPU**.
2. Leave `PYFEAT_MODE = 'modern'` (Cell 1) and **Run all**.
3. If Cell 3 says py-feat FAILED, set `PYFEAT_MODE = 'pinned'` in Cell 1, **Runtime -> Restart runtime**, then Run all again.
4. Read the verdict in the last cell.


## Cell 1 — Config

In [ ]:
# Path B (default): current py-feat on Colab's modern stack — NO downgrade.
# Path A: downgrade to the pinned .venv-feat stack (torch 2.0.1+cu118). Needs a
#         Runtime RESTART after install for the torch downgrade to take effect.
PYFEAT_MODE = "modern"   # "modern" (Path B)  |  "pinned" (Path A)

N_BENCH   = 12           # clips to benchmark
AU_TOP_K  = 12           # AU runs on the top-K frames by conf x sharpness
DRIVE_ROOT        = "/content/drive/MyDrive/DeepSentinel_data"
DRIVE_SEG_ARCHIVE = DRIVE_ROOT + "/segments.zip"   # a few clips are sampled from here
REPO_URL    = "https://github.com/gjvlio/emotion-based-multimodal-deepfake-detector.git"
REPO_BRANCH = "feat/mosei-preprocess-and-eval"

# Full-dataset sizes for the extrapolation (clips needing AU-on z_v):
TOTALS = [6277, 14240, 18369]   # MOSEI-only, current cache, all-sources


## Cell 2 — Install py-feat (Path B or A) + visual-pipeline deps

In [ ]:
import subprocess, sys
def sh(c): print('$', c); return subprocess.run(c, shell=True).returncode
if PYFEAT_MODE == "modern":
    print('Path B: current py-feat on the modern Colab stack (no downgrade)...')
    sh('pip install -q py-feat')
elif PYFEAT_MODE == "pinned":
    print('Path A: downgrading to the pinned .venv-feat stack (torch 2.0.1+cu118)...')
    sh('pip install -q numpy==1.23.5 scipy==1.10.1 scikit-learn==1.2.2 scikit-image==0.20.0 pandas==1.5.3')
    sh('pip install -q torch==2.0.1+cu118 torchvision==0.15.2+cu118 torchaudio==2.0.2+cu118 --index-url https://download.pytorch.org/whl/cu118')
    sh('pip install -q py-feat kornia==0.7.0 nilearn==0.10.2')
else:
    raise ValueError("PYFEAT_MODE must be 'modern' or 'pinned'")
# deps the visual branch needs (ViT + insightface); harmless if already present
sh('pip install -q transformers timm insightface onnxruntime-gpu opencv-python-headless librosa soundfile')
if PYFEAT_MODE == "pinned":
    print('\n*** If torch was downgraded: Runtime -> RESTART RUNTIME now, then re-run from Cell 3. ***')
print('install done.')

## Cell 3 — Verify torch/CUDA + py-feat Detector loads

In [ ]:
import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch:', torch.__version__, '| CUDA avail:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
try:
    from feat import Detector
    _ = Detector(device=dev)
    PYFEAT_WORKS = True
    print('py-feat OK — Detector loaded on', dev, '=> this Colab stack CAN run AU-on.')
except Exception as e:
    PYFEAT_WORKS = False
    print('py-feat FAILED:', type(e).__name__, '-', e)
    print("\n-> If PYFEAT_MODE='modern' failed: set it to 'pinned' in Cell 1, RESTART RUNTIME, re-run.")
    print("-> If 'pinned' also failed on CUDA: torch 2.0.1+cu118 may not run on this T4 driver "
          '(then AU-on stays a LOCAL .venv-feat job).')
assert PYFEAT_WORKS, 'py-feat could not load — see guidance above.'

## Cell 4 — Mount Drive, clone repo, grab a few sample clips

In [ ]:
from google.colab import drive
import os, zipfile
from pathlib import Path
drive.mount('/content/drive')
assert os.path.exists(DRIVE_SEG_ARCHIVE), f'segments.zip not found at {DRIVE_SEG_ARCHIVE}'
REPO_DIR = '/content/thesis'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git','clone','--branch',REPO_BRANCH,'--depth','1',REPO_URL,REPO_DIR], check=True)
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
SAMPLE_DIR = '/content/au_sample'; os.makedirs(SAMPLE_DIR, exist_ok=True)
with zipfile.ZipFile(DRIVE_SEG_ARCHIVE) as z:
    names = [n for n in z.namelist() if n.lower().endswith('.mp4')][:N_BENCH]
    for n in names: z.extract(n, SAMPLE_DIR)
clips = sorted(Path(SAMPLE_DIR).rglob('*.mp4'))
print(f'{len(clips)} sample clips ready; repo @ {REPO_BRANCH}')

## Cell 5 — Benchmark AU-on `z_v` (non-destructive)

Calls `get_z_v(..., AU-on)` on each clip and times it. First clip is a warm-up (model downloads + first-call overhead) and is excluded from the average.

In [ ]:
import time, statistics as st
from src.preprocessing import visual
visual.configure_au(enabled=True, device=dev, top_k=AU_TOP_K)
from src.preprocessing.visual import get_z_v

def extract(v):
    return get_z_v(str(v), vit_model_name='google/vit-base-patch16-224', detector='retinaface',
                   n_keyframes=8, frame_size=224, target_fps=25.0, motion_threshold=0.3,
                   confidence_threshold=0.7, device=dev)

print('warming up (ViT + insightface + py-feat first call)...')
_ = extract(clips[0])
print('warm — benchmarking', len(clips)-1, 'clips:\n')
times = []
for v in clips[1:]:
    t0 = time.time(); z = extract(v); dt = time.time() - t0; times.append(dt)
    print(f'  {v.name[:42]:<42} {dt:6.1f}s   z_v={tuple(z.shape)}')
mean = st.mean(times); med = st.median(times)
print(f'\nAU-on z_v per clip:  mean={mean:.1f}s   median={med:.1f}s   (n={len(times)})')

## Cell 6 — Verdict: does Colab work, and how long for the full run?

In [ ]:
print('='*64)
print(f'  py-feat on this Colab stack : {"WORKS" if PYFEAT_WORKS else "FAILED"}  (mode={PYFEAT_MODE})')
print(f'  AU-on z_v per clip (median) : {med:.1f}s')
print('='*64)
for total in TOTALS:
    hrs = total * med / 3600
    print(f'  {total:>6} clips  ->  {hrs:6.1f} GPU-hours  (~{hrs/24:.1f} days on ONE T4)')
    print(f'              4-way shard  ->  ~{hrs/4:.1f}h wall-clock across 4 T4s')
print()
print('Reminder: z_at is REUSED (AU-independent), so the real run rebuilds ONLY z_v.')
print('Real run (LOCAL or, if this passed, sharded on Colab):')
print('  preprocess_all.py --use_au --au_top_k 12 \\')
print('     --out_dir data/preprocessed_au_on --reuse_zat_from data/preprocessed')
print('  (--out_dir is REQUIRED with --use_au — the AU-off baseline is never overwritten.)')